##### After random forest the results were not good so i wanted to try xgboost whether it will perform better than RF

In [62]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import roc_auc_score, roc_curve,precision_recall_curve
from imblearn.over_sampling import SMOTE
import joblib 
import os 

X_train = pd.read_csv('../Data/X_train.csv')
X_test = pd.read_csv('../Data/X_test.csv')
Y_train = np.load('../Data/Y_train.npy', allow_pickle=True)
Y_test = np.load('../Data/Y_test.npy', allow_pickle=True)

print("Data loaded")
print(f"X_train shape: {X_train.shape}")

Data loaded
X_train shape: (402698, 28)


In [63]:
Y_train_mapped = pd.Series(Y_train)
Y_test_mapped = pd.Series(Y_test)
print(Y_train_mapped.value_counts())
print(Y_test_mapped.value_counts())

2    308748
1     87961
0      5989
Name: count, dtype: int64
2    77187
1    21991
0     1497
Name: count, dtype: int64


In [64]:
smote = SMOTE(sampling_strategy={0: 30000}, random_state=42) 
X_train_res, Y_train_res = smote.fit_resample(X_train, Y_train_mapped)

In [65]:
print(pd.Series(Y_train_res).value_counts().sort_index())

0     30000
1     87961
2    308748
Name: count, dtype: int64


In [66]:
print("Before:", pd.Series(Y_train_mapped).value_counts().sort_index())
print("After:", pd.Series(Y_train_res).value_counts().sort_index())

Before: 0      5989
1     87961
2    308748
Name: count, dtype: int64
After: 0     30000
1     87961
2    308748
Name: count, dtype: int64


In [67]:
X_tr,X_val,Y_tr,Y_val = train_test_split(X_train_res,Y_train_res,test_size=0.2,random_state=42,stratify=Y_train_res)
model_xgb = XGBClassifier(
    n_estimators=3000,
    early_stopping_rounds=50,
    learning_rate=0.1,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    num_class=3,
    objective='multi:softprob',
    eval_metric='mlogloss',
    n_jobs=-1
)



In [68]:
model_xgb.fit(X_tr,Y_tr,eval_set=[(X_val,Y_val)],verbose=10)

[0]	validation_0-mlogloss:0.72466
[10]	validation_0-mlogloss:0.64610
[20]	validation_0-mlogloss:0.61786
[30]	validation_0-mlogloss:0.60089
[40]	validation_0-mlogloss:0.58771
[50]	validation_0-mlogloss:0.57851
[60]	validation_0-mlogloss:0.57114
[70]	validation_0-mlogloss:0.56573
[80]	validation_0-mlogloss:0.56120
[90]	validation_0-mlogloss:0.55747
[100]	validation_0-mlogloss:0.55355
[110]	validation_0-mlogloss:0.55047
[120]	validation_0-mlogloss:0.54772
[130]	validation_0-mlogloss:0.54578
[140]	validation_0-mlogloss:0.54365
[150]	validation_0-mlogloss:0.54230
[160]	validation_0-mlogloss:0.54113
[170]	validation_0-mlogloss:0.53994
[180]	validation_0-mlogloss:0.53897
[190]	validation_0-mlogloss:0.53821
[200]	validation_0-mlogloss:0.53745
[210]	validation_0-mlogloss:0.53656
[220]	validation_0-mlogloss:0.53607
[230]	validation_0-mlogloss:0.53570
[240]	validation_0-mlogloss:0.53503
[250]	validation_0-mlogloss:0.53469
[260]	validation_0-mlogloss:0.53436
[270]	validation_0-mlogloss:0.53390
[28

,"objective objective: str | xgboost.objective.Objective | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.8
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",50
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegr

In [69]:
y_pred_xgb = model_xgb.predict(X_test) 

In [70]:
print(classification_report(Y_test, y_pred_xgb, target_names=['Fatal','Serious','Slight']))

              precision    recall  f1-score   support

       Fatal       0.33      0.00      0.01      1497
     Serious       0.50      0.05      0.08     21991
      Slight       0.77      0.99      0.87     77187

    accuracy                           0.77    100675
   macro avg       0.54      0.35      0.32    100675
weighted avg       0.71      0.77      0.68    100675



In [71]:
y_proba_xgb = model_xgb.predict_proba(X_test)
print("First 10 predicted probabilities:\n", y_proba_xgb[:10])
print("Mean predicted P(Fatal):", y_proba_xgb[:, 0].mean())
print("Max predicted P(Fatal):", y_proba_xgb[:, 0].max())

First 10 predicted probabilities:
 [[1.2119502e-03 9.2501216e-02 9.0628684e-01]
 [3.6960468e-02 4.9124905e-01 4.7179055e-01]
 [2.9850862e-04 2.4170059e-01 7.5800091e-01]
 [2.8227018e-03 1.4680497e-01 8.5037231e-01]
 [6.2326579e-03 3.2179606e-01 6.7197126e-01]
 [1.4178968e-03 1.4218774e-01 8.5639435e-01]
 [2.5028655e-02 3.8915586e-01 5.8581549e-01]
 [6.6973232e-03 1.5219963e-01 8.4110308e-01]
 [5.8862609e-03 2.9510888e-01 6.9900483e-01]
 [1.0220941e-02 2.5077015e-01 7.3900896e-01]]
Mean predicted P(Fatal): 0.01607541
Max predicted P(Fatal): 0.5581316


In [72]:
y_true_fatal = (Y_test ==0).astype(int)
y_score_fatal = y_proba_xgb[:, 0]
precision, recall, thresholds = precision_recall_curve(y_true_fatal, y_score_fatal)
target_recall =0.6
valid_idx= np.where(recall >= target_recall)[0]
idx = valid_idx[-1]

print(f"Threshold for recall>={target_recall}: {thresholds[idx]:.4f}")
print(f"At that threshold — precision: {precision[idx]:.4f}, recall: {recall[idx]:.4f}")


Threshold for recall>=0.6: 0.0255
At that threshold — precision: 0.0503, recall: 0.6005


In [73]:
threshold = 0.0255
y_proba = model_xgb.predict_proba(X_test)
y_pred_final = np.where(y_proba[:, 0] >= threshold, 0, np.argmax(y_proba[:, 1:], axis=1) + 1)
print(classification_report(Y_test_mapped, y_pred_final, target_names=['Fatal','Serious','Slight']))

              precision    recall  f1-score   support

       Fatal       0.05      0.60      0.09      1497
     Serious       0.45      0.01      0.01     21991
      Slight       0.80      0.85      0.82     77187

    accuracy                           0.66    100675
   macro avg       0.43      0.49      0.31    100675
weighted avg       0.71      0.66      0.64    100675



In [74]:
threshold_fatal = 0.0255
not_fatal_mask = y_proba[:, 0] < threshold_fatal

In [75]:
tuning_mask = not_fatal_mask & (Y_test_mapped != 0)
y_true_serious = (Y_test_mapped[tuning_mask] == 1).astype(int)

p_serious = y_proba[tuning_mask, 1]
p_slight = y_proba[tuning_mask, 2]
score_serious = p_serious / (p_serious + p_slight)  # normalized, ignores Fatal prob mass

from sklearn.metrics import precision_recall_curve
precision2, recall2, thresholds2 = precision_recall_curve(y_true_serious, score_serious)

target_recall2 = 0.4  # start here — Serious isn't as cost-asymmetric as Fatal, don't copy 0.6 blindly
valid_idx2 = np.where(recall2 >= target_recall2)[0]
idx2 = valid_idx2[-1]
threshold_serious = thresholds2[idx2]
print(f"threshold_serious: {threshold_serious:.4f}, precision: {precision2[idx2]:.4f}, recall: {recall2[idx2]:.4f}")

threshold_serious: 0.2407, precision: 0.2969, recall: 0.4000


In [76]:
p_serious_norm = y_proba[:, 1] / (y_proba[:, 1] + y_proba[:, 2])
final_pred = np.where(
    y_proba[:, 0] >= threshold_fatal, 0,
    np.where(p_serious_norm >= threshold_serious, 1, 2)
)
print(classification_report(Y_test_mapped, final_pred, target_names=['Fatal','Serious','Slight']))

              precision    recall  f1-score   support

       Fatal       0.05      0.60      0.09      1497
     Serious       0.29      0.29      0.29     21991
      Slight       0.84      0.66      0.74     77187

    accuracy                           0.58    100675
   macro avg       0.39      0.52      0.37    100675
weighted avg       0.70      0.58      0.63    100675



##### I optimized for Fatal-class recall at the cost of Serious-class recall and overall precision, because in this domain missing a fatal accident is worse than a false alarm — this tradeoff came at a real cost to the Serious class, which I did not separately optimize for.

In [77]:
target_recall2 = 0.25
valid_idx2 = np.where(recall2 >= target_recall2)[0]
idx2 = valid_idx2[-1]
threshold_serious = thresholds2[idx2]
print(f"threshold_serious: {threshold_serious:.4f}, precision: {precision2[idx2]:.4f}, recall: {recall2[idx2]:.4f}")

p_serious_norm = y_proba[:, 1] / (y_proba[:, 1] + y_proba[:, 2])
final_pred = np.where(
    y_proba[:, 0] >= threshold_fatal, 0,
    np.where(p_serious_norm >= threshold_serious, 1, 2)
)
print(classification_report(Y_test_mapped, final_pred, target_names=['Fatal','Serious','Slight']))

threshold_serious: 0.2811, precision: 0.3240, recall: 0.2500
              precision    recall  f1-score   support

       Fatal       0.05      0.60      0.09      1497
     Serious       0.32      0.18      0.23     21991
      Slight       0.82      0.75      0.78     77187

    accuracy                           0.62    100675
   macro avg       0.40      0.51      0.37    100675
weighted avg       0.70      0.62      0.65    100675



In [80]:
os.makedirs('../Models', exist_ok=True)
joblib.dump(model_xgb, '../Models/xgb_model.pkl')
joblib.dump(list(X_train.columns), '../Models/feature_columns.pkl')

['../Models/feature_columns.pkl']

In [79]:
print(list(X_train.columns))

['longitude', 'latitude', 'number_of_vehicles', 'number_of_casualties', 'day_of_week', 'first_road_class', 'road_type', 'speed_limit', 'junction_detail', 'second_road_class', 'pedestrian_crossing', 'light_conditions', 'weather_conditions', 'road_surface_conditions', 'special_conditions_at_site', 'carriageway_hazards', 'urban_or_rural_area', 'hour', 'month', 'season', 'is_rush_hour', 'is_weekend', 'is_dark', 'is_bad_weather', 'is_highspeed', 'is_urban', 'is_junction', 'is_hazards']


In [84]:
import joblib
scaler = joblib.load(r'C:\Users\pch37\Documents\Uk-road-accident-severity-prediction\Models\scaler.pkl')
print("mean_:", scaler.mean_)
print("scale_:", scaler.scale_)

import numpy as np
test_row = np.array([[-1.5, 52.5, 30, 2, 14, 8, 1]])
print("transformed:", scaler.transform(test_row))

mean_: [-1.2053374  52.36323573 35.99682641  1.8281988  13.75819348  6.68323409
  1.27221392]
scale_: [ 1.3578256   1.3170262  14.23295046  0.68474281  5.14225352  3.41587297
  0.69648962]
transformed: [[-0.21701064  0.10384324 -0.42133403  0.25089887  0.04702345  0.38548445
  -0.390837  ]]


c:\Users\pch37\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [83]:
fatal_rows = X_train[Y_train_mapped == 0]
print(fatal_rows[['longitude','latitude','speed_limit','number_of_vehicles',
                   'number_of_casualties','hour','month']].describe())

         longitude     latitude  speed_limit  number_of_vehicles  \
count  5989.000000  5989.000000  5989.000000         5989.000000   
mean     -0.289272     0.334031     0.761311           -0.087561   
std       1.087563     1.202058     1.116088            1.368793   
min      -4.151724    -1.777403    -1.123929           -1.209503   
25%      -1.020696    -0.615952    -0.421334           -1.209503   
50%      -0.239391     0.130107     0.983856            0.250899   
75%       0.596944     0.920068     1.686451            0.250899   
max       2.167709     5.990148     2.389046           20.696532   

       number_of_casualties         hour        month  
count           5989.000000  5989.000000  5989.000000  
mean               0.538854    -0.105102     0.024739  
std                1.733106     1.224655     1.001493  
min               -0.390837    -2.675518    -1.663772  
25%               -0.390837    -0.925313    -0.785519  
50%               -0.390837     0.047023     0.0927